In [16]:
import pandas as pd, si_units as si, numpy as np, feos, matplotlib.pyplot as plt, os
from molmass import Formula
from ctREFPROP.ctREFPROP import REFPROPFunctionLibrary

In [17]:
os.environ['RPPREFIX'] = r"/home/darshan/Software/REFPROP/REFPROP-cmake/build"
RP = REFPROPFunctionLibrary(os.environ['RPPREFIX'])
RP.SETPATHdll(os.environ['RPPREFIX'])
MOLAR_BASE_SI = RP.GETENUMdll(0, "MOLAR BASE SI").iEnum

In [8]:
He = Formula("He")
Ne = Formula("Ne")
molar_masses = np.array([He.mass, Ne.mass])  # g/mol
print(f"He molar mass: {He.mass} g/mol")
print(f"Ne molar mass: {Ne.mass} g/mol")

He molar mass: 4.002602 g/mol
Ne molar mass: 20.1797 g/mol


In [9]:
# Load SAFT-VRQ-Mie parameters
parameters = feos.Parameters.from_json(
    ["helium", "neon"], 
    "parameters.json",
    binary_path="aasen2020_binary.json"
)

# Create equation of state
saftvrqmie = feos.EquationOfState.saftvrqmie(parameters)

parameters

|component|molarweight|m|sigma|epsilon_k|lr|la|fh|
|-|-|-|-|-|-|-|-|
|helium|4.002601643881807|1.0|2.7443|5.4195|9.0|6.0|1|
|neon|20.17969806457545|1.0|2.7778|37.501|13.0|6.0|1|

|component 1|component 2|k_ij|l_ij|
|-|-|-|-|
|helium|neon|-0.22|0.0|

In [10]:
# Define conditions
z = np.array([0.5, 0.5])  # 50-50 He-Ne mixture
T = 35.0                   # K
P = 30e5                   # Pa (30 bar)

# Create state
state = feos.State(
    saftvrqmie, 
    temperature=T * si.KELVIN, 
    pressure=P * si.PASCAL, 
    molefracs=z
)

# Get residual entropy
s_res = state.molar_entropy(feos.Contributions.Residual)

print(f"Temperature: {T} K")
print(f"Pressure: {P/1e5} bar")
print(f"Composition (He/Ne): {z}")
print(f"Residual molar entropy: {s_res}")

Temperature: 35.0 K
Pressure: 30.0 bar
Composition (He/Ne): [0.5 0.5]
Residual molar entropy: -2.871833646296655  J/mol/K


In [11]:
# Pure helium
z_he = np.array([1.0, 0.0])
state_he = feos.State(saftvrqmie, temperature=T*si.KELVIN, pressure=P*si.PASCAL, molefracs=z_he)
s_res_he = state_he.molar_entropy(feos.Contributions.Residual)
print(f"Pure He residual entropy: {s_res_he}")

# Pure neon
z_ne = np.array([0.0, 1.0])
state_ne = feos.State(saftvrqmie, temperature=T*si.KELVIN, pressure=P*si.PASCAL, molefracs=z_ne)
s_res_ne = state_ne.molar_entropy(feos.Contributions.Residual)
print(f"Pure Ne residual entropy: {s_res_ne}")

Pure He residual entropy: -1.3051221568350975  J/mol/K
Pure Ne residual entropy: -19.229308021299815  J/mol/K


In [18]:
# Test conditions (same as before)
T_test = 35.0  # K
P_test = 30e5  # Pa (30 bar)

# Pure Helium
r_he = RP.REFPROPdll("HELIUM", "TP", "S;S0", MOLAR_BASE_SI, 0, 0, T_test, P_test, [1.0])
s_total_he_rp = r_he.Output[0]  # J/mol/K
s_ideal_he_rp = r_he.Output[1]  # J/mol/K
s_res_he_rp = s_total_he_rp - s_ideal_he_rp

# Extract numeric value from FeOS (divide by the unit)
s_res_he_value = float(s_res_he / (si.JOULE / si.MOL / si.KELVIN))

print("Pure Helium Validation:")
print(f"  FeOS (SAFT-VRQ-Mie): {s_res_he_value:.4f} J/mol/K")
print(f"  RefProp:             {s_res_he_rp:.4f} J/mol/K")
print(f"  Absolute difference: {abs(s_res_he_value - s_res_he_rp):.4f} J/mol/K")
print(f"  Relative difference: {100*abs(s_res_he_value - s_res_he_rp)/abs(s_res_he_rp):.2f}%")
print()

# Pure Neon
r_ne = RP.REFPROPdll("NEON", "TP", "S;S0", MOLAR_BASE_SI, 0, 0, T_test, P_test, [1.0])
s_total_ne_rp = r_ne.Output[0]
s_ideal_ne_rp = r_ne.Output[1]
s_res_ne_rp = s_total_ne_rp - s_ideal_ne_rp

# Extract numeric value from FeOS
s_res_ne_value = float(s_res_ne / (si.JOULE / si.MOL / si.KELVIN))

print("Pure Neon Validation:")
print(f"  FeOS (SAFT-VRQ-Mie): {s_res_ne_value:.4f} J/mol/K")
print(f"  RefProp:             {s_res_ne_rp:.4f} J/mol/K")
print(f"  Absolute difference: {abs(s_res_ne_value - s_res_ne_rp):.4f} J/mol/K")
print(f"  Relative difference: {100*abs(s_res_ne_value - s_res_ne_rp)/abs(s_res_ne_rp):.2f}%")

Pure Helium Validation:
  FeOS (SAFT-VRQ-Mie): -1.3051 J/mol/K
  RefProp:             -1.3675 J/mol/K
  Absolute difference: 0.0624 J/mol/K
  Relative difference: 4.56%

Pure Neon Validation:
  FeOS (SAFT-VRQ-Mie): -19.2293 J/mol/K
  RefProp:             -18.7414 J/mol/K
  Absolute difference: 0.4879 J/mol/K
  Relative difference: 2.60%


In [19]:
# Get transport properties from RefProp for pure components

# Pure Helium transport properties
r_he_visc = RP.REFPROPdll("HELIUM", "TP", "VIS;TCX", MOLAR_BASE_SI, 0, 0, T_test, P_test, [1.0])
eta_he_rp = r_he_visc.Output[0]  # Pa·s
lambda_he_rp = r_he_visc.Output[1]  # W/(m·K)

print("Pure Helium Transport Properties (RefProp):")
print(f"  Viscosity:            {eta_he_rp*1e6:.4f} µPa·s")
print(f"  Thermal conductivity: {lambda_he_rp*1e3:.4f} mW/(m·K)")
print()

# Pure Neon transport properties
r_ne_visc = RP.REFPROPdll("NEON", "TP", "VIS;TCX", MOLAR_BASE_SI, 0, 0, T_test, P_test, [1.0])
eta_ne_rp = r_ne_visc.Output[0]  # Pa·s
lambda_ne_rp = r_ne_visc.Output[1]  # W/(m·K)

print("Pure Neon Transport Properties (RefProp):")
print(f"  Viscosity:            {eta_ne_rp*1e6:.4f} µPa·s")
print(f"  Thermal conductivity: {lambda_ne_rp*1e3:.4f} mW/(m·K)")
print()

# He-Ne mixture (50-50)
r_mix = RP.REFPROPdll("HELIUM;NEON", "TP", "VIS;TCX", MOLAR_BASE_SI, 0, 0, T_test, P_test, [0.5, 0.5])
eta_mix_rp = r_mix.Output[0]  # Pa·s
lambda_mix_rp = r_mix.Output[1]  # W/(m·K)

print("He-Ne Mixture (50-50) Transport Properties (RefProp):")
print(f"  Viscosity:            {eta_mix_rp*1e6:.4f} µPa·s")
print(f"  Thermal conductivity: {lambda_mix_rp*1e3:.4f} mW/(m·K)")

Pure Helium Transport Properties (RefProp):
  Viscosity:            5.7865 µPa·s
  Thermal conductivity: 43.9131 mW/(m·K)

Pure Neon Transport Properties (RefProp):
  Viscosity:            72.1707 µPa·s
  Thermal conductivity: 75.9589 mW/(m·K)

He-Ne Mixture (50-50) Transport Properties (RefProp):
  Viscosity:            -9999990000000.0000 µPa·s
  Thermal conductivity: -9999990000.0000 mW/(m·K)
